In [3]:
# Importing our needed methods
from pyspark.sql.functions import col, to_date

# Read stores data
df_stores = spark.read.format("csv").option("header", True).option("inferSchema", True).load("Files/bronze/stores_products_data/Stores_Catalog.csv")

# Read products data
df_products = spark.read.format("csv").option("header", True).option("inferSchema", True).load("Files/bronze/stores_products_data/Products_Master.csv")
df_products = df_products.withColumn("UnitPrice", col("UnitPrice").cast("decimal(10,2)"))

# Read header data
df_invoices_header = spark.read.format("delta").load("Files/bronze/pos_retail_transactions/Invoices_Header")
df_january_headers = df_invoices_header.where(to_date(col("InvoiceDate")) >= "2026-01-01").where(to_date(col("InvoiceDate")) < "2026-02-01")

# View Creation
df_january_headers.createOrReplaceTempView("january_headers")

# Read line data
df_january_lines = spark.sql(
""" 
    SELECT
        *
    FROM delta.`Files/bronze/pos_retail_transactions/Invoices_Lines`
    WHERE InvoiceID IN(SELECT InvoiceID FROM january_headers)
"""                         )

# View Creation
df_january_lines.createOrReplaceTempView("january_lines")

# Initializing control schema that contain control table for maintaining pipeline watermark
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS control
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS control.pipeline_state (
        PipelineName STRING,
        PipelineWatermark TIMESTAMP,
        UpdatedAt TIMESTAMP
    )
    USING DELTA
""")

StatementMeta(, ffa7d1ad-364a-4045-bdd9-964901e08773, 5, Finished, Available, Finished, False)

DataFrame[]

In [4]:
# Initial load for stores
df_stores.write.format("delta").mode("Overwrite").saveAsTable("silver.stores")

# Initial load for products
df_products.write.format("delta").mode("Overwrite").saveAsTable("silver.products")

# Initial load for headers and quarantine headers
spark.sql(
"""
    CREATE OR REPLACE TABLE silver.invoices_header AS
    SELECT *
    FROM january_headers AS h
    WHERE EXISTS (
        SELECT 1
        FROM january_lines AS l
        WHERE l.InvoiceID = h.InvoiceID
            AND l.Quantity IS NOT NULL
                )
"""      )

spark.sql(
"""
CREATE OR REPLACE TABLE silver.quarantine_invoices_header AS
SELECT *,
    'No valid invoice lines' AS QuarantineReason,
    CURRENT_TIMESTAMP() AS QuarantinedAt
FROM january_headers AS h
WHERE NOT EXISTS (
    SELECT 1
    FROM january_lines AS l
    WHERE l.InvoiceID = h.InvoiceID
        AND l.Quantity IS NOT NULL
                )
"""      )

# Initial load for lines and quarantine lines
spark.sql(
"""
    CREATE OR REPLACE TABLE silver.invoices_lines AS
    SELECT
        *
    FROM january_lines
    WHERE Quantity IS NOT NULL
"""      )

spark.sql(
"""
    CREATE OR REPLACE TABLE silver.quarantine_invoices_lines AS
    SELECT
        *,
        'Missing Quantity' AS QuarantineReason,
        CURRENT_TIMESTAMP() AS QuarantinedAt
    FROM january_lines
    WHERE Quantity IS NULL
"""      )

# Updating the pipeline watermark
# Why done like this?
    # To make it work correctly if we ran nb_bronze_to_silver and ran the initial again so the pipline watermark be updated correctly
    # Prevent re-insertion for bronze_to_silver record when initial script ran twice
spark.sql(
"""
    MERGE INTO control.pipeline_state AS target
    USING (
        SELECT
            'bronze_to_silver' AS PipelineName,
            MAX(InvoiceDate) AS PipelineWatermark
        FROM january_headers
    ) AS source
    ON target.PipelineName = source.PipelineName

    WHEN MATCHED THEN
        UPDATE SET
            target.PipelineWatermark = source.PipelineWatermark,
            target.UpdatedAt = current_timestamp()

    WHEN NOT MATCHED THEN
        INSERT (PipelineName, PipelineWatermark, UpdatedAt)
        VALUES (
            source.PipelineName,
            source.PipelineWatermark,
            current_timestamp()
        );
"""     )

StatementMeta(, ffa7d1ad-364a-4045-bdd9-964901e08773, 6, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [5]:
%%sql
-- Validation
SELECT
    'Clean Lines' AS Category,
     COUNT(*) AS TotalCount
FROM silver.invoices_lines

UNION ALL

SELECT 
    'Quarantine Lines',
    COUNT(*) 
 FROM silver.quarantine_invoices_lines

 UNION ALL

 SELECT 
    'Clean Header',
    COUNT(*) 
 FROM silver.invoices_header

 UNION ALL
 
 SELECT 
    'Quarantine Header',
    COUNT(*) 
 FROM silver.quarantine_invoices_header

UNION ALL

SELECT 
    'Product Count',
    COUNT(*)
FROM silver.products

UNION ALL

SELECT 
    'Stores Count',
    COUNT(*)
FROM silver.stores

UNION ALL

-- Check that the Header–Line relationship remains logically consistent after transformation
SELECT 
    'Lines Without Header',
    COUNT(*)
FROM silver.invoices_lines as l
LEFT JOIN silver.invoices_header as h
ON l.InvoiceID = h.InvoiceID
WHERE h.InvoiceID IS NULL

StatementMeta(, ffa7d1ad-364a-4045-bdd9-964901e08773, 7, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 2 fields>

In [6]:
%%sql
SELECT * FROM control.pipeline_state

StatementMeta(, ffa7d1ad-364a-4045-bdd9-964901e08773, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>